# PDP-CR on Google Colab

Run GA, TSR and MLS on Google's CPU runtime. Time limits per seed: **50 tasks: 10 minutes; 100: 30 minutes; 200: 60 minutes**. Each method uses five seeds. CPLEX is excluded.

Run the cells in order and allow Google Drive access when prompted. Code and completed seed results stay in Drive, so rerunning with the same settings resumes completed work. An interrupted seed restarts from the beginning. The full campaign is long; you can select one size or family below and use a different result name for each selection.


In [ ]:
SIZES = [50, 100, 200]
FAMILIES = ["LC1", "LR1", "LRC1"]
SEEDS = [0, 1, 2, 3, 4]
SECONDS_BY_N = {50: 10 * 60.0, 100: 30 * 60.0, 200: 60 * 60.0}
WORKERS = 2
GA_POPULATION = 16
RESULT_NAME = "colab_9_10_30_60min"


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/PDPCR_Pilot_GA_TSR_MLS_colab")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/dhducpro/PDPCR_Pilot_GA_TSR_MLS.git", str(ROOT)], check=True)
if not (ROOT / "run_pilot.py").is_file():
    raise FileNotFoundError("Project code not found in " + str(ROOT))
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
for name in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy>=1.24,<3", "pandas>=2.0", "matplotlib>=3.7"], check=True)
OUTPUT = ROOT / "results" / RESULT_NAME
print("Cloud project:", ROOT)
print("Results:", OUTPUT)
print("Workers:", WORKERS, "| Reported CPUs:", os.cpu_count())
from math import ceil
hours = len(FAMILIES) * 3 * ceil(len(SEEDS)/WORKERS) * sum(SECONDS_BY_N[n] for n in SIZES) / 3600
print(f"Scheduled budget: about {hours:g} hours plus setup and overhead; actual CPU speed and availability vary.")


In [ ]:
from prepare_instances import prepare

prepared = prepare(sizes=SIZES, families=FAMILIES, robots=10,
                   requirement_seed=0, download=True)
manifest_path = ROOT / "instances" / "manifest.json"
print("Prepared instances:", len(prepared))


In [ ]:
from run_pilot import run_pilot

comparison = run_pilot(sizes=SIZES, families=FAMILIES,
                       seeds=SEEDS, seconds_by_n=SECONDS_BY_N,
                       workers=WORKERS, population=GA_POPULATION,
                       methods=("GA", "TSR", "MLS"), manifest=manifest_path,
                       output_dir=OUTPUT, resume=True)


In [ ]:
import pandas as pd

results = pd.read_csv(OUTPUT / "comparison.csv")
columns = ["n", "family", "GA_best_of_5", "TSR_best_of_5", "MLS_best_of_5",
           "GA_improvement_vs_TSR_pct", "GA_improvement_vs_MLS_pct"]
display(results[columns].round(4))
print("Positive percentage = GA has a lower makespan.")
display(results.groupby("n")[["GA_improvement_vs_TSR_pct", "GA_improvement_vs_MLS_pct"]].mean().round(4))
print("Saved in Google Drive:", OUTPUT)
